In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType
from pyspark.sql import functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_location")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/hospital_data/resource_type=Location/"
silver_base_path = "../../data_lake/silver/silver_location/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])


telecom_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
        StructField("use", StringType(), True)
])

address_schema = StructType([
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])

position_schema = StructType([
        StructField("longitude", DoubleType(), True),
        StructField("latitude", DoubleType(), True),
    ])

identifier_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True)
    ])

managing_organization_schema = StructType([
        StructField("identifier", identifier_schema, True),
        StructField("display", StringType(), True)
    ])

In [ ]:
def telecom_value(system_value):
    matches = F.filter(
        col("telecom"),
        lambda x: x["system"] == system_value
    )
    
    first_match = F.try_element_at(matches, F.lit(1))   # null if no match
    return first_match["value"]

In [ ]:
df_location = spark.read.format("parquet").load(bronze_path)

In [ ]:
df_inter1 = (
    df_location.select(
        col("resource.id").alias("location_id"),
        col("resource.status").alias("status"),
        col("resource.name").alias("name"),
        from_json(col("resource.telecom"),ArrayType(telecom_schema)).alias("telecom"),
        from_json(col("resource.address"), address_schema).alias("address"),
        from_json(col("resource.position"), position_schema).alias("position"),
        from_json(col("resource.managingOrganization"), managing_organization_schema).alias("managing_organization"),
        col("input_file_name")
    )
)

In [ ]:
df_inter2 = (
    df_inter1.select(
        col("location_id"),
        col("status"),
        col("name"),
        telecom_value("phone").alias("phone"),
        telecom_value("email").alias("email"),
        F.concat_ws(", ", col("address")["line"]).alias("address_line"),
        col("address")["city"].alias("city"),
        col("address")["state"].alias("state"),
        col("address")["postalCode"].alias("postalCode"),
        col("address")["country"].alias("country"),
        col("position.latitude").alias("latitude"),
        col("position.longitude").alias("longitude"),
        col("managing_organization.identifier.value").alias("managing_organization_id"),
        col("managing_organization.display").alias("managing_organization_display"),
        col("input_file_name"),
        current_timestamp().alias("silver_timestamp")
    )
)

In [ ]:
df_inter2.write.mode("overwrite").format("parquet").save(silver_base_path)

In [ ]:
spark.stop()